In [1]:
import os
os.makedirs("models", exist_ok=True)
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.initializers import GlorotUniform, HeNormal
import tensorflow as tf
import random
import itertools
import gc
from tensorflow.keras import backend as K

SEED = 72
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values
y_train_full = train_df[target].values
x_test = test_df[features].values
y_test = test_df[target].values

mean = x_train_full.mean(axis=0)
std = x_train_full.std(axis=0)
x_train = (x_train_full - mean) / std
x_test = (x_test - mean) / std

def build_model(
    layers_config,
    activations,
    initializers,
    optimizer_name,
    learning_rate,
    dropout_rates,
    use_batch_norm,
    kernel_regularizer,
    bias_regularizer
):
    model = Sequential()
    for i, (units, act, init) in enumerate(zip(layers_config, activations, initializers)):
        if i == 0:
            model.add(Dense(units,
                            activation=act,
                            input_shape=(x_train.shape[1],),
                            kernel_initializer=init,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        else:
            model.add(Dense(units,
                            activation=act,
                            kernel_initializer=init,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        if use_batch_norm[i]:
            model.add(BatchNormalization())
        if dropout_rates[i] > 0:
            model.add(Dropout(dropout_rates[i]))
    model.add(Dense(1, activation='linear', kernel_initializer=GlorotUniform(seed=SEED)))
    
    if optimizer_name == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer")
    
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

def get_initializer_for_activation(activation):
    if activation in ('relu', 'elu'):
        initializer = np.random.choice([HeNormal(seed=SEED), HeUniform(seed=SEED)])
        return initializer
    elif activation == 'tanh':
        initializer = np.random.choice([GlorotNormal(seed=SEED), GlorotUniform(seed=SEED)])
        return initializer
    else:

        return GlorotUniform(seed=SEED)

def generate_architectures(depth):
    architectures = []
    base_units = [64, 32, 16, 8]
    
    # Генерация последовательно уменьшающихся архитектур
    for start in base_units:
        arch = []
        current = start
        for _ in range(depth):
            arch.append(current)
            current = max(8, current // 2)
        architectures.append(tuple(arch))
    
    # Генерация сбалансированных архитектур
    for units in [32, 16]:
        architectures.append(tuple([units] * depth))
    
    # Удаление дубликатов и сортировка по сложности
    architectures = list(set(architectures))
    architectures.sort(key=lambda x: sum(x), reverse=True)
    return architectures

# Параметры перебора
TARGET_MAE = 37000
MAX_EXPERIMENTS_PER_DEPTH = 50
ACTIVATIONS = ['relu', 'elu', 'tanh']
OPTIMIZERS = ['adam', 'rmsprop']
LEARNING_RATES = [1e-3, 5e-4, 1e-4]
BATCH_SIZES = [64, 32, 16]
DROPOUT_RATES = [0.0, 0.1]
USE_BATCH_NORM_OPTIONS = [True, False]
REGULARIZERS = [None, l1(1e-4), l2(1e-4)]

results = []
early_stop = EarlyStopping(monitor='val_mae', patience=50, restore_best_weights=True)
experiment_id = 1
target_reached = False

for depth in [2, 3, 4]:
    if target_reached:
        break
        
    architectures = generate_architectures(depth)
    print(f"\nГлубина сети: {depth}, Количество архитектур: {len(architectures)}")
    
    for arch in architectures:
        if target_reached:
            break
            
        # Генерация всех комбинаций активаций для текущей глубины
        activation_combinations = list(itertools.product(ACTIVATIONS, repeat=depth))
        
        for acts in activation_combinations:
            if target_reached or experiment_id > MAX_EXPERIMENTS_PER_DEPTH * depth:
                break
                
            initializers = [get_initializer_for_activation(act) for act in acts]
            
            # Генерация гиперпараметров через декартово произведение
            hyperparams = itertools.product(
                OPTIMIZERS,
                LEARNING_RATES,
                BATCH_SIZES,
                [tuple([bool(i < depth-1) for i in range(depth)])],  # BatchNorm на всех кроме последнего
                [tuple([0.0 if i == depth-1 else np.random.choice(DROPOUT_RATES) for i in range(depth)])],  # Dropout кроме выходного
                REGULARIZERS,
                REGULARIZERS
            )
            
            for (opt_name, lr, batch_size, bn_flags, dropout_rates, k_reg, b_reg) in hyperparams:
                if target_reached:
                    break
                    
                print(f"\nЭксперимент {experiment_id}: Глубина={depth}, Архитектура={arch}, Активации={acts}")
                print(f"Гиперпараметры: оптимизатор={opt_name}, lr={lr}, batch={batch_size}")
                
                try:
                    model = build_model(
                        layers_config=arch,
                        activations=acts,
                        initializers=initializers,
                        optimizer_name=opt_name,
                        learning_rate=lr,
                        dropout_rates=dropout_rates,
                        use_batch_norm=bn_flags,
                        kernel_regularizer=k_reg,
                        bias_regularizer=b_reg
                    )
                    
                    history = model.fit(
                        x_train, y_train_full,
                        validation_split=0.2,
                        epochs=300,
                        batch_size=batch_size,
                        callbacks=[early_stop],
                        verbose=1
                    )
                    
                    actual_epochs = len(history.history['mae'])
                    train_mae = history.history['mae'][-1]
                    val_mae = history.history['val_mae'][-1]
                    test_mae = model.evaluate(x_test, y_test, verbose=0)[1]
                    
                    print(f"Результаты: train_mae={train_mae:.2f}, val_mae={val_mae:.2f}, test_mae={test_mae:.2f}")
                    
                    # Сохранение модели
                    model_path = f"models/model_depth{depth}_id{experiment_id}.keras"
                    model.save(model_path)
                    
                    results.append({
                        'ID': experiment_id,
                        'depth': depth,
                        'layers': str(arch),
                        'activations': str(acts),
                        'optimizer': opt_name,
                        'learning_rate': lr,
                        'batch_size': batch_size,
                        'epochs': actual_epochs,
                        'dropout': str(dropout_rates),
                        'batch_norm': str(bn_flags),
                        'kernel_regularizer': str(k_reg),
                        'bias_regularizer': str(b_reg),
                        'train_mae': train_mae,
                        'val_mae': val_mae,
                        'test_mae': test_mae,
                        'model_path': model_path
                    })
                    
                    if test_mae <= TARGET_MAE:
                        print(f"\nЦЕЛЬ ДОСТИГНУТА! MAE = {test_mae:.2f} <= {TARGET_MAE}")
                        target_reached = True
                        
                        # Сохранение таблицы результатов
                        pd.DataFrame(results).to_csv("final_results.csv", index=False)
                        break
                    
                    # Промежуточное сохранение каждые 5 экспериментов
                    if experiment_id % 5 == 0:
                        pd.DataFrame(results).to_csv(f"intermediate_results_{experiment_id}.csv", index=False)
                        print(f"Промежуточные результаты сохранены (эксперимент {experiment_id})")
                    
                    experiment_id += 1
                    
                    # Очистка памяти
                    del model
                    K.clear_session()
                    gc.collect()
                    
                except Exception as e:
                    print(f"Ошибка в эксперименте {experiment_id}: {str(e)}")
                    continue

# Финальное сохранение результатов
pd.DataFrame(results).to_csv("all_experiments_results.csv", index=False)
print("\nВсе эксперименты завершены. Результаты сохранены в all_experiments_results.csv")

if target_reached:
    print(f"Лучшая модель достигла целевого MAE={TARGET_MAE} и сохранена в файлы в папке models/")
else:
    print(f"Цель MAE≤{TARGET_MAE} не достигнута. Лучший результат: {min([r['test_mae'] for r in results]):.2f}")

2025-11-04 18:09:42.559651: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-04 18:09:42.647847: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 18:09:45.243946: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 18:09:48.398953: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur